In [1]:
import pandas as pd
import altair as alt

In [7]:
df2 = pd.read_csv("201608-hubway-tripdata.csv")
df2['starttime'] = pd.to_datetime(df2['starttime'])
df2['hour'] = df2['starttime'].dt.hour
df2['weekday'] = df2['starttime'].dt.day_name()
df2['trip_minutes'] = df2['tripduration'] / 60
day_order = ['Monday','Tuesday','Wednesday','Thursday',
             'Friday','Saturday','Sunday']

In [42]:
# Tested but not used on website
agg_df = (
    df2.groupby(['hour', 'weekday'])
    .size()
    .reset_index(name='trip_count')
)

heatmap1 = alt.Chart(agg_df).mark_rect().encode(
    x=alt.X('hour:O', title='Hour of Day'),
    y=alt.Y('weekday:N', sort=day_order, title='Day of Week'),
    color=alt.Color(
        'trip_count:Q',
        scale=alt.Scale(scheme='blues'),
        title='Number of Trips'
    )
).properties(
    title='Bluebikes Trips by Hour and Day of Week',
    width=700, height=400
)
heatmap1.show()

alt.Chart(...)

In [43]:
# Also used for testing, but not used
df2['user_label'] = df2['usertype'].map({'Subscriber': 'Member', 'Customer': 'Non-Member'})
agg_df = (
    df2.groupby(['hour', 'weekday', 'user_label'])
    .size()
    .reset_index(name='trip_count')
)

color_scale = alt.Scale(
    scheme='blues',
    domain=[0, agg_df['trip_count'].max()]
)

def make_heatmap(user_label):
    return alt.Chart(agg_df).mark_rect().transform_filter(
        alt.datum.user_label == user_label
    ).encode(
        x=alt.X('hour:O', title='Hour of Day'),
        y=alt.Y('weekday:N', sort=day_order, title='Day of Week'),
        color=alt.Color(
            'trip_count:Q',
            scale=color_scale,
            title='Number of Trips'
        ),
        tooltip=[
            alt.Tooltip('weekday:N', title='Day'),
            alt.Tooltip('hour:O', title='Hour'),
            alt.Tooltip('trip_count:Q', title='Trips', format=',')
        ]
    ).properties(
        title=user_label,
        width=500,
        height=350
    )

heatmap_members     = make_heatmap('Member')
heatmap_nonmembers  = make_heatmap('Non-Member')

heatmap_combined = alt.hconcat(
    heatmap_members,
    heatmap_nonmembers
).properties(
    title=alt.TitleParams(
        'Bluebikes Trips by Hour and Day of Week',
        anchor='middle'
    )
)

heatmap_combined.show()

alt.HConcatChart(...)

In [24]:
import json

In [44]:
# Heatmap used
df2['user_label']       = df2['usertype'].map({'Subscriber': 'Member', 'Customer': 'Non-Member'})
df2['gender_label']     = df2['gender'].map({0: 'Unknown', 1: 'Male', 2: 'Female'})
df2['birth_year_clean'] = pd.to_numeric(df2['birth year'], errors='coerce')
df2['age']              = 2016 - df2['birth_year_clean']

def age_group(a):
    if pd.isna(a) or a < 10 or a > 100: return 'Unknown'
    if a <= 21:  return 'Gen Z (21 and under)'
    if a <= 36:  return 'Millennial (22-36)'
    if a <= 52:  return 'Gen X (37-52)'
    return 'Boomer+ (53+)'

df2['age_group'] = df2['age'].apply(age_group)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

def make_agg(df, group_col, view_name):
    agg = df.groupby(['hour', 'weekday', group_col]).size().reset_index(name='trip_count')
    agg.columns = ['hour', 'weekday', 'group', 'trip_count']
    agg['view'] = view_name
    return agg

overall          = df2.groupby(['hour', 'weekday']).size().reset_index(name='trip_count')
overall['group'] = 'All Trips'
overall['view']  = 'Overall'

all_agg = pd.concat([
    overall,
    make_agg(df2, 'user_label',   'Member Type'),
    make_agg(df2, 'gender_label', 'Gender'),
    make_agg(df2, 'age_group',    'Age Group'),
], ignore_index=True)

day_totals            = all_agg.groupby(['view', 'group', 'weekday'])['trip_count'].transform('sum')
all_agg['pct_of_day'] = (all_agg['trip_count'] / day_totals * 100).round(1)
global_max            = int(all_agg['trip_count'].max())

VIEW_CONFIGS = {
    'Overall':     {
        'groups': ['All Trips'],
        'width': 820, 'height': 260
    },
    'Member Type': {
        'groups': ['Member', 'Non-Member'],
        'width': 440, 'height': 260
    },
    'Gender':      {
        'groups': ['Male', 'Female', 'Unknown'],
        'width': 300, 'height': 260
    },
    'Age Group':   {
        'groups': ['Gen Z (21 and under)', 'Millennial (22-36)',
                   'Gen X (37-52)', 'Boomer+ (53+)'],
        'width': 225, 'height': 260
    },
}

def make_panel(subset, group_name, width, height, is_first, is_last):
    return (
        alt.Chart(subset[subset['group'] == group_name])
        .mark_rect(stroke='white', strokeWidth=0.5)
        .encode(
            x=alt.X('hour:O', title='Hour of Day',
                    axis=alt.Axis(labelAngle=0, values=list(range(0, 24, 6)))),
            y=alt.Y('weekday:N', sort=day_order,
                    title='Day of Week' if is_first else None,
                    axis=alt.Axis(labels=is_first, ticks=is_first)),
            color=alt.Color(
                'trip_count:Q',
                scale=alt.Scale(scheme='blues', domain=[0, global_max]),
                title='# Trips',
                legend=alt.Legend(gradientLength=150) if is_last else None,
            ),
            tooltip=[
                alt.Tooltip('group:N',      title='Group'),
                alt.Tooltip('weekday:N',    title='Day'),
                alt.Tooltip('hour:O',       title='Hour'),
                alt.Tooltip('trip_count:Q', title='Trips',            format=','),
                alt.Tooltip('pct_of_day:Q', title="% of day's trips", format='.1f'),
            ]
        )
        .properties(
            width=width, height=height,
            title=alt.TitleParams(group_name, fontSize=13, fontWeight='bold', anchor='middle')
        )
    )

def make_view_chart(view_name, config):
    groups  = config['groups']
    subset  = all_agg[all_agg['view'] == view_name]
    panels  = [
        make_panel(subset, g, config['width'], config['height'],
                   is_first=(i == 0), is_last=(i == len(groups) - 1))
        for i, g in enumerate(groups)
    ]
    return (
        alt.hconcat(*panels, spacing=12)
        .configure_view(stroke=None)
        .configure_axis(labelFontSize=11, titleFontSize=12, grid=False)
    )

charts = {name: make_view_chart(name, cfg) for name, cfg in VIEW_CONFIGS.items()}

default_view = 'Member Type'
view_names   = list(VIEW_CONFIGS.keys())

radio_html = ' &nbsp; '.join(
    f'<label><input type="radio" name="view" value="{v}"'
    f'{" checked" if v == default_view else ""}> {v}</label>'
    for v in view_names
)

divs_html = '\n'.join(
    f'<div class="cv {"active" if v == default_view else ""}" '
    f'id="cv-{i}"><div id="chart-{i}"></div></div>'
    for i, v in enumerate(view_names)
)

embeds_js = '\n'.join(
    f'vegaEmbed("#chart-{i}", {json.dumps(c.to_dict())}, {{actions: false}});'
    for i, c in enumerate(charts.values())
)

# html for plot
html = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Bluebikes Heatmaps</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <style>
    body      {{ font-family: sans-serif; padding: 24px; background: #fafafa; }}
    h2        {{ margin: 0 0 4px; font-size: 20px; }}
    .subtitle {{ color: #888; font-size: 13px; margin-bottom: 18px; }}
    .controls {{ margin-bottom: 20px; font-size: 14px; }}
    .controls strong {{ margin-right: 10px; }}
    label     {{ margin-right: 16px; cursor: pointer; }}
    .cv       {{ display: none; }}
    .cv.active{{ display: block; }}
  </style>
</head>
<body>
  <h2>Bluebikes Trips by Hour and Day of Week</h2>
  <p class="subtitle">Hover any cell for trip count and % of that day's total</p>
  <div class="controls">
    <strong>Split by:</strong> {radio_html}
  </div>
  {divs_html}
  <script>
    {embeds_js}
    document.querySelectorAll('input[name="view"]').forEach((r, idx) => {{
      r.addEventListener('change', () => {{
        document.querySelectorAll('.cv').forEach(el => el.classList.remove('active'));
        document.getElementById('cv-' + idx).classList.add('active');
      }});
    }});
  </script>
</body>
</html>"""

with open('heatmap2.html', 'w', encoding='utf-8') as f:
    f.write(html)
print("Saved → heatmap1.html")

Saved → heatmap1.html


In [45]:
# Bar chart for testing
start_counts = df2['start station name'].value_counts()
end_counts = df2['end station name'].value_counts()

station_total = start_counts.add(end_counts, fill_value=0).reset_index()
station_total.columns = ['station', 'total_trips']

top_stations = station_total.sort_values(by='total_trips', ascending=False).head(10)

barchart1 = alt.Chart(top_stations).mark_bar().encode(
    y=alt.Y('station:N', sort='-x', title='Station'),
    x=alt.X('total_trips:Q', title='Total Trips (Start + End)')
).properties(
    title='Top 10 Most Active Stations',
    width=500, height=200
)

barchart1.show()

alt.Chart(...)

In [47]:
# Barchart used on website
n = len(station_total)
station_total = station_total.sort_values('total_trips', ascending=False).reset_index(drop=True)
station_total['rank_top']    = range(1, n + 1)
station_total['rank_bottom'] = range(n, 0, -1)

slider = alt.param(
    name='top_n',
    value=10,
    bind=alt.binding_range(min=5, max=50, step=1, name='Number of Stations: ')
)

mode_param = alt.param(
    name='mode',
    value='top',
    bind=alt.binding_radio(
        options=['top', 'bottom'],
        labels=['Most Active', 'Least Active'],
        name='Show: '
    )
)

base = (
    alt.Chart(station_total)
    .transform_filter(
        "(mode == 'top'    && datum.rank_top    <= top_n) ||"
        "(mode == 'bottom' && datum.rank_bottom <= top_n)"
    )
    .transform_calculate(
        sort_val    = "mode == 'top' ? -datum.total_trips : datum.total_trips",
        rank_label  = "mode == 'top' ? '#' + datum.rank_top : 'Bottom #' + datum.rank_bottom",
        color_val   = "mode == 'top' ? datum.total_trips : -datum.total_trips"
    )
    .add_params(slider, mode_param)
)

x_enc = alt.X(
    'station:N',
    sort=alt.EncodingSortField(field='sort_val', order='ascending'),
    title='Station',
    axis=alt.Axis(labelAngle=-35, labelLimit=220)
)

bars = base.mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
    x=x_enc,
    y=alt.Y('total_trips:Q', title='Total Trips'),
    color=alt.Color(
        'color_val:Q',
        scale=alt.Scale(scheme='redblue', domain=[-station_total['total_trips'].max(),
                                                   station_total['total_trips'].max()]),
        legend=None
    ),
    opacity=alt.value(0.85),
    tooltip=[
        alt.Tooltip('station:N',     title='Station'),
        alt.Tooltip('total_trips:Q', title='Total Trips', format=','),
        alt.Tooltip('rank_label:N',  title='Rank'),
    ]
)

avg_rule = base.mark_rule(
    color='#e45c2b',
    strokeDash=[6, 4],
    strokeWidth=2
).encode(
    y=alt.Y('mean(total_trips):Q'),
    tooltip=[alt.Tooltip('mean(total_trips):Q', title='Avg Trips (shown)', format=',.0f')]
)

avg_label = base.mark_text(
    align='left', dx=4, dy=-8,
    fontSize=11, fontWeight='bold', color='#e45c2b'
).encode(
    y=alt.Y('mean(total_trips):Q'),
    x=alt.value(0),
    text=alt.value('― avg')
)

bar_labels = base.mark_text(dy=-6, fontSize=10, color='#333').encode(
    x=x_enc,
    y=alt.Y('total_trips:Q'),
    text=alt.Text('total_trips:Q', format='~s')
)

barchart1 = (bars + avg_rule + avg_label + bar_labels).properties(
    title=alt.TitleParams(
        'Bluebikes Station Activity',
        subtitle='Use the slider to change number of stations • Toggle to switch between most & least active',
        anchor='middle',
        subtitleColor='#888',
        subtitleFontSize=12
    ),
    width=alt.Step(48),
    height=420
)

barchart1.show()
barchart1.save('barchart2.html')

alt.LayerChart(...)

In [48]:
# CandleChart
df2['starttime']    = pd.to_datetime(df2['starttime'])
df2['hour']         = df2['starttime'].dt.hour
df2['day']          = df2['starttime'].dt.day
df2['trip_minutes'] = df2['tripduration'] / 60
df2['user_group']   = df2['usertype'].map({'Subscriber': 'Member', 'Customer': 'Non-Member'})

day_labels = {
    1:'Mon',2:'Tue',3:'Wed',4:'Thu',5:'Fri',6:'Sat',7:'Sun',
    8:'Mon',9:'Tue',10:'Wed',11:'Thu',12:'Fri',13:'Sat',14:'Sun',
    15:'Mon',16:'Tue',17:'Wed',18:'Thu',19:'Fri',20:'Sat',21:'Sun',
    22:'Mon',23:'Tue',24:'Wed',25:'Thu',26:'Fri',27:'Sat',28:'Sun',
    29:'Mon',30:'Tue',31:'Wed'
}
weekend_days = {6, 7, 13, 14, 20, 21, 27, 28}

def box_stats_group(series):
    q1  = series.quantile(0.25)
    med = series.quantile(0.50)
    q3  = series.quantile(0.75)
    iqr = q3 - q1
    return pd.Series({
        'q1':           round(q1, 2),
        'median':       round(med, 2),
        'q3':           round(q3, 2),
        'mean':         round(series.mean(), 2),
        'whisker_low':  round(max(series.min(), q1 - 1.5 * iqr), 2),
        'whisker_high': round(min(series.max(), q3 + 1.5 * iqr), 2),
        'p95':          round(series.quantile(0.95), 2),
        'count':        int(len(series))
    })

records = []
for dim, col in [('day', 'day'), ('hour', 'hour')]:
    for ug in ['All', 'Member', 'Non-Member']:
        sub = df2 if ug == 'All' else df2[df2['user_group'] == ug]
        for xval, grp in sub.groupby(col)['trip_minutes']:
            row = box_stats_group(grp).to_dict()
            row['dim']        = dim
            row['user_group'] = ug
            row['x_val']      = int(xval)
            row['day_label']  = day_labels.get(int(xval), '') if dim == 'day' else f"{int(xval):02d}:00"
            row['is_weekend'] = int(xval) in weekend_days if dim == 'day' else False
            records.append(row)

stats_df = pd.DataFrame(records)

ycap_param = alt.param(
    name='ycap',
    value=45,
    bind=alt.binding_range(min=15, max=120, step=1, name='Y-axis cap (min): ')
)

dim_param = alt.param(
    name='dim_sel',
    value='day',
    bind=alt.binding_radio(
        options=['day', 'hour'],
        labels=['Day of Month', 'Hour of Day'],
        name='X axis: '
    )
)

ug_param = alt.param(
    name='ug_sel',
    value='All',
    bind=alt.binding_radio(
        options=['All', 'Member', 'Non-Member'],
        name='User group: '
    )
)

base = (
    alt.Chart(stats_df)
    .add_params(ycap_param, dim_param, ug_param)
    .transform_filter('datum.dim === dim_sel && datum.user_group === ug_sel')
    .transform_calculate(
        whi_c  = 'min(datum.whisker_high, ycap)',
        q3_c   = 'min(datum.q3,           ycap)',
        med_c  = 'min(datum.median,        ycap)',
        mean_c = 'min(datum.mean,          ycap)',
        q1_c   = 'min(datum.q1,            ycap)',
        wlo_c  = 'min(datum.whisker_low,   ycap)',
        clipped = 'datum.whisker_high > ycap || datum.mean > ycap'
    )
)

y_scale  = alt.Scale(domainMin=0)

stroke_color = alt.Color(
    'is_weekend:N',
    scale=alt.Scale(domain=[False, True], range=['#4f8ef7', '#f59e0b']),
    legend=None
)
fill_color = alt.Color(
    'is_weekend:N',
    scale=alt.Scale(domain=[False, True], range=['#1d3461', '#78350f']),
    legend=None
)

x_enc = alt.X(
    'x_val:O',
    title='Day of Month / Hour of Day',
    axis=alt.Axis(labelAngle=0)
)

tooltip_fields = [
    alt.Tooltip('x_val:O',         title='X'),
    alt.Tooltip('count:Q',         title='Rides',         format=','),
    alt.Tooltip('whisker_low:Q',   title='Whisker low',   format='.1f'),
    alt.Tooltip('q1:Q',            title='Q1',            format='.1f'),
    alt.Tooltip('median:Q',        title='Median',        format='.1f'),
    alt.Tooltip('mean:Q',          title='Mean (actual)', format='.1f'),
    alt.Tooltip('q3:Q',            title='Q3',            format='.1f'),
    alt.Tooltip('whisker_high:Q',  title='Whisker high',  format='.1f'),
    alt.Tooltip('p95:Q',           title='95th pct',      format='.1f'),
]

whiskers = base.mark_rule(strokeWidth=1.5, opacity=0.6).encode(
    x=x_enc,
    y=alt.Y('wlo_c:Q',  title='Trip Duration (min)', scale=y_scale),
    y2=alt.Y2('whi_c:Q'),
    stroke=stroke_color
)

boxes = base.mark_bar(width=16, cornerRadiusTopLeft=3, cornerRadiusTopRight=3).encode(
    x=x_enc,
    y=alt.Y('q1_c:Q',  scale=y_scale),
    y2=alt.Y2('q3_c:Q'),
    fill=fill_color,
    stroke=stroke_color,
    strokeWidth=alt.value(1.5),
    tooltip=tooltip_fields
)

medians = base.mark_tick(color='#e11d48', thickness=2, width=20).encode(
    x=x_enc,
    y=alt.Y('med_c:Q', scale=y_scale)
)

means = base.mark_point(shape='diamond', filled=True, size=50, color='#f0f4ff').encode(
    x=x_enc,
    y=alt.Y('mean_c:Q', scale=y_scale)
)

clipped_markers = (
    base
    .transform_filter('datum.clipped === true')
    .mark_point(shape='triangle-up', filled=True, color='#ef4444', size=70, opacity=0.85)
    .encode(
        x=x_enc,
        y=alt.Y('ycap:Q', scale=y_scale),
        tooltip=[
            alt.Tooltip('x_val:O',        title='X'),
            alt.Tooltip('mean:Q',          title='Actual mean',         format='.1f'),
            alt.Tooltip('whisker_high:Q',  title='Actual whisker high', format='.1f'),
        ]
    )
)

cap_rule = base.mark_rule(
    color='#ef4444', strokeDash=[4, 3], strokeWidth=1, opacity=0.45
).encode(
    y=alt.Y('ycap:Q', scale=y_scale)
)

chart = (
    (whiskers + boxes + medians + means + clipped_markers + cap_rule)
    .properties(
        width=860,
        height=400,
        title=alt.TitleParams(
            'Bluebikes Trip Duration',
            subtitle=[
                'Box = Q1–Q3  |  Red tick = median  |  Diamond = mean  |  Whiskers = 1.5×IQR',
                'Amber = weekend  |  Blue = weekday'
            ],
            anchor='middle',
            fontSize=17,
            subtitleColor='#888',
            subtitleFontSize=11,
            subtitlePadding=6
        )
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12,
        gridColor='#e5e7eb',
        domainColor='#d1d5db'
    )
    .configure_view(stroke=None)
)

chart.show()
chart.save('candle1.html')

alt.LayerChart(...)

In [49]:
# Chord graph copied from groupmate
import numpy as np

df = pd.read_csv("201608-hubway-tripdata.csv")
start = df['start station name'].value_counts()
end   = df['end station name'].value_counts()
total = start.add(end, fill_value=0).sort_values(ascending=False)
top_stations = total.head(20).index.tolist()

def shorten(name):
    replacements = {
        'Boston Public Library - 700 Boylston St.':               'BPL Boylston',
        'MIT at Mass Ave / Amherst St':                           'MIT Mass Ave',
        'Harvard Square at Mass Ave/ Dunster':                    'Harvard Sq',
        'South Station - 700 Atlantic Ave.':                      'South Station',
        'Central Square at Mass Ave / Essex St':                  'Central Sq',
        'MIT Stata Center at Vassar St / Main St':                'MIT Stata',
        'Beacon St / Mass Ave':                                   'Beacon/Mass Ave',
        'Charles Circle - Charles St. at Cambridge St.':          'Charles Circle',
        'Boylston St. at Arlington St.':                          'Boylston/Arlington',
        'The Esplanade - Beacon St. at Arlington St.':            'Esplanade',
        'Mayor Martin J Walsh - 28 State St':                     '28 State St',
        'Cross St. at Hanover St.':                               'Cross/Hanover St',
        'Kenmore Sq / Comm Ave':                                  'Kenmore Sq',
        'TD Garden - West End Park':                              'TD Garden',
        'Back Bay / South End Station':                           'Back Bay Station',
        'One Kendall Square at Hampshire St / Portland St':       'One Kendall Sq',
        'Christian Science Plaza':                                'Christian Science',
        'Cambridge St. at Joy St.':                               'Cambridge/Joy St',
        'Boylston at Fairfield':                                  'Boylston/Fairfield',
    }
    return replacements.get(name, name[:20] + '…' if len(name) > 20 else name)

short_names = [shorten(s) for s in top_stations]

sub_all   = df[df['start station name'].isin(top_stations) &
               df['end station name'].isin(top_stations)]
flows_all = sub_all.groupby(['start station name', 'end station name']).size().reset_index(name='count')

matrices = {}
for n in range(10, 21):
    stations_n = top_stations[:n]
    idx = {s: i for i, s in enumerate(stations_n)}
    mat = np.zeros((n, n), dtype=int)
    for _, row in flows_all.iterrows():
        if row['start station name'] in idx and row['end station name'] in idx:
            mat[idx[row['start station name']]][idx[row['end station name']]] += row['count']
    matrices[n] = mat.tolist()

groups = {
    'Cambridge/MIT':   ['MIT Mass Ave','Harvard Sq','Central Sq','MIT Stata',
                        'Beacon/Mass Ave','One Kendall Sq','Cambridge/Joy St'],
    'Downtown Boston': ['South Station','28 State St','Cross/Hanover St','TD Garden'],
    'Back Bay':        ['BPL Boylston','Boylston/Arlington','Esplanade','Back Bay Station',
                        'Boylston/Fairfield','Christian Science','Kenmore Sq'],
    'Charles River':   ['Charles Circle'],
    'Kendall':         ['Kendall T'],
}
group_colors = {
    'Cambridge/MIT':   '#4f8ef7',
    'Downtown Boston': '#f59e0b',
    'Back Bay':        '#10b981',
    'Charles River':   '#a78bfa',
    'Kendall':         '#fb7185',
    'Other':           '#94a3b8',
}
station_group = {s: g for g, slist in groups.items() for s in slist}

station_meta = [
    {
        'full':  full,
        'short': short,
        'group': station_group.get(short, 'Other'),
        'color': group_colors[station_group.get(short, 'Other')],
        'total': int(total[full])
    }
    for full, short in zip(top_stations, short_names)
]

data_js = json.dumps({'matrices': matrices, 'stations': station_meta, 'groupColors': group_colors})

# html needed for plot
html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Bluebikes Station Flow</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.8.5/d3.min.js"></script>
<style>
* { box-sizing:border-box; margin:0; padding:0; }
body {
  font-family: 'Segoe UI', system-ui, sans-serif;
  background: #0c0e1a; color: #e2e8f0;
  min-height: 100vh; padding: 24px 32px;
}
h2 { font-size:22px; font-weight:700; color:#f0f4ff; margin-bottom:4px; }
.subtitle { font-size:13px; color:#64748b; margin-bottom:20px; }

.controls {
  display:flex; flex-wrap:wrap; gap:24px; align-items:flex-end;
  background:#141627; border:1px solid #1e2235; border-radius:10px;
  padding:14px 20px; margin-bottom:24px;
}
.control-group { display:flex; flex-direction:column; gap:6px; }
.control-label { font-size:11px; font-weight:600; text-transform:uppercase; letter-spacing:.8px; color:#64748b; }
.btn-group { display:flex; gap:6px; }
.btn {
  padding:5px 13px; border:1px solid #1e2235; border-radius:6px;
  background:#0c0e1a; color:#64748b; font-size:13px; cursor:pointer; transition:all .15s;
}
.btn:hover  { border-color:#4f8ef7; color:#c7d6ff; }
.btn.active { background:#1a2744; border-color:#4f8ef7; color:#93c5fd; font-weight:600; }

.slider-row { display:flex; align-items:center; gap:10px; }
input[type=range] {
  -webkit-appearance:none; width:180px; height:4px;
  background:#1e2235; border-radius:2px; outline:none;
}
input[type=range]::-webkit-slider-thumb {
  -webkit-appearance:none; width:14px; height:14px;
  border-radius:50%; background:#4f8ef7; cursor:pointer;
}
.slider-val { font-size:13px; font-weight:600; color:#93c5fd; min-width:36px; }

.main { display:flex; gap:24px; align-items:flex-start; }
#chart-area { flex-shrink:0; }
.sidebar { flex:1; min-width:200px; }

.legend-title { font-size:12px; font-weight:600; text-transform:uppercase; letter-spacing:.7px; color:#64748b; margin-bottom:10px; }
.legend-item  { display:flex; align-items:center; gap:8px; margin-bottom:7px; font-size:13px; color:#94a3b8; cursor:pointer; transition:opacity .15s; }
.legend-item:hover { color:#e2e8f0; }
.legend-swatch { width:12px; height:12px; border-radius:3px; flex-shrink:0; }

.stats { margin-top:24px; }
.stats-title { font-size:12px; font-weight:600; text-transform:uppercase; letter-spacing:.7px; color:#64748b; margin-bottom:10px; }
.stat-card {
  background:#141627; border:1px solid #1e2235; border-radius:8px;
  padding:12px 14px; margin-bottom:8px; font-size:13px; display:none;
}
.stat-card.visible { display:block; }
.stat-name  { font-weight:700; color:#93c5fd; margin-bottom:6px; font-size:14px; }
.stat-row   { display:flex; justify-content:space-between; line-height:1.9; }
.stat-key   { color:#64748b; }
.stat-val   { font-weight:600; color:#e2e8f0; }
.stat-hint  { font-size:11px; color:#334155; margin-top:6px; font-style:italic; }

#tooltip {
  position:fixed; pointer-events:none; z-index:999;
  background:#141627; border:1px solid #4f8ef7; border-radius:8px;
  padding:10px 14px; font-size:13px; line-height:1.8; color:#cbd5e1;
  box-shadow:0 8px 32px rgba(0,0,0,.6); display:none; max-width:240px;
}
#tooltip .tt-title { font-weight:700; color:#93c5fd; margin-bottom:3px; }
</style>
</head>
<body>

<h2>Bluebikes Station-to-Station Flow</h2>
<p class="subtitle">Ribbons show trips between station pairs — width encodes volume — colored by the source station's neighborhood</p>

<div class="controls">
  <div class="control-group">
    <span class="control-label">Top N Stations</span>
    <div class="slider-row">
      <input type="range" id="n-slider" min="10" max="20" step="1" value="15">
      <span class="slider-val" id="n-val">15</span>
    </div>
  </div>
  <div class="control-group">
    <span class="control-label">Color by</span>
    <div class="btn-group" id="color-btns">
      <button class="btn active" data-val="group">Neighborhood</button>
      <button class="btn"        data-val="volume">Trip Volume</button>
    </div>
  </div>
  <div class="control-group">
    <span class="control-label">Self-loops</span>
    <div class="btn-group" id="loop-btns">
      <button class="btn"        data-val="true">Show</button>
      <button class="btn active" data-val="false">Hide</button>
    </div>
  </div>
</div>

<div class="main">
  <div id="chart-area"></div>
  <div class="sidebar">
    <div class="legend-title">Neighborhoods</div>
    <div id="legend"></div>
    <div class="stats">
      <div class="stats-title">Station Detail</div>
      <div class="stat-card" id="stat-card">
        <div class="stat-name"  id="sc-name"></div>
        <div class="stat-row"><span class="stat-key">Total trips</span>     <span class="stat-val" id="sc-total"></span></div>
        <div class="stat-row"><span class="stat-key">Trips out</span>        <span class="stat-val" id="sc-out"></span></div>
        <div class="stat-row"><span class="stat-key">Trips in</span>         <span class="stat-val" id="sc-in"></span></div>
        <div class="stat-row"><span class="stat-key">Top destination</span>  <span class="stat-val" id="sc-dest"></span></div>
        <div class="stat-row"><span class="stat-key">Top origin</span>       <span class="stat-val" id="sc-orig"></span></div>
        <div class="stat-hint">Click arc again or background to deselect</div>
      </div>
      <div id="stat-hint-default" style="font-size:12px;color:#334155;font-style:italic;">
        Hover a ribbon to see flow details.<br>Click an arc to pin station stats.
      </div>
    </div>
  </div>
</div>

<div id="tooltip"></div>

<script>
const RAW = """ + data_js + """;

const W = 580, H = 580, R = 220, rInner = R - 22;
const state = { n:15, colorBy:'group', showLoops:false, pinned:null };
const volScale = d3.scaleSequential(d3.interpolatePlasma).domain([0, 1]);

function getColor(i, matrix, mode) {
  if (mode === 'group') return RAW.stations[i].color;
  const rowSum = d3.sum(matrix[i]);
  const total  = d3.sum(matrix.flat());
  return volScale(rowSum / total * state.n);
}

// Legend
const legendEl = document.getElementById('legend');
let currentHighlightGroup = null;
Object.entries(RAW.groupColors).forEach(([grp, col]) => {
  const item = document.createElement('div');
  item.className = 'legend-item';
  item.innerHTML = `<div class="legend-swatch" style="background:${col}"></div>${grp}`;
  item.addEventListener('click', () => {
    if (currentHighlightGroup === grp) { currentHighlightGroup = null; }
    else { currentHighlightGroup = grp; state.pinned = null; }
    applyOpacity();
  });
  legendEl.appendChild(item);
});

function applyOpacity() {
  d3.selectAll('.arc-path').attr('opacity', (d, i) => {
    if (state.pinned !== null) return i === state.pinned ? 1 : 0.12;
    if (currentHighlightGroup) return RAW.stations[i].group === currentHighlightGroup ? 1 : 0.2;
    return 0.9;
  });
  d3.selectAll('.ribbon').attr('opacity', d => {
    const si = d.source.index, ti = d.target.index;
    if (state.pinned !== null) return (si === state.pinned || ti === state.pinned) ? 0.75 : 0.04;
    if (currentHighlightGroup) {
      return (RAW.stations[si].group === currentHighlightGroup ||
              RAW.stations[ti].group === currentHighlightGroup) ? 0.7 : 0.04;
    }
    return 0.65;
  });
}

function showStatCard(i, matrix, stations) {
  const st       = stations[i];
  const outV     = matrix[i];
  const inV      = matrix.map(r => r[i]);
  const outTotal = d3.sum(outV);
  const inTotal  = d3.sum(inV);
  const topDestIdx = outV.indexOf(Math.max(...outV.map((v, j) => j === i ? -1 : v)));
  const topOrigIdx = inV.indexOf( Math.max(...inV.map((v, j)  => j === i ? -1 : v)));

  document.getElementById('sc-name').textContent  = st.short;
  document.getElementById('sc-total').textContent = st.total.toLocaleString() + ' trips';
  document.getElementById('sc-out').textContent   = outTotal.toLocaleString();
  document.getElementById('sc-in').textContent    = inTotal.toLocaleString();
  document.getElementById('sc-dest').textContent  = stations[topDestIdx]?.short + ` (${outV[topDestIdx].toLocaleString()})`;
  document.getElementById('sc-orig').textContent  = stations[topOrigIdx]?.short + ` (${inV[topOrigIdx].toLocaleString()})`;
  document.getElementById('stat-card').classList.add('visible');
  document.getElementById('stat-hint-default').style.display = 'none';
}

const tt = document.getElementById('tooltip');
function showTooltip(e, html) {
  tt.innerHTML = html;
  tt.style.display = 'block';
  tt.style.left = Math.min(e.clientX + 14, window.innerWidth - 260) + 'px';
  tt.style.top  = (e.clientY + 14) + 'px';
}
function hideTooltip() { tt.style.display = 'none'; }

function render() {
  d3.select('#chart-area').selectAll('*').remove();
  document.getElementById('stat-card').classList.remove('visible');
  document.getElementById('stat-hint-default').style.display = 'block';

  const n        = state.n;
  const stations = RAW.stations.slice(0, n);
  let matrix     = RAW.matrices[n].map(row => row.slice(0, n));
  if (!state.showLoops) matrix = matrix.map((row, i) => row.map((v, j) => i === j ? 0 : v));

  const chord  = d3.chord().padAngle(0.03).sortSubgroups(d3.descending)(matrix);
  const arc    = d3.arc().innerRadius(rInner).outerRadius(R);
  const ribbon = d3.ribbon().radius(rInner - 2);

  const svg = d3.select('#chart-area').append('svg').attr('width', W).attr('height', H);
  svg.on('click', e => {
    if (e.target.tagName === 'svg') {
      state.pinned = null; currentHighlightGroup = null;
      document.getElementById('stat-card').classList.remove('visible');
      document.getElementById('stat-hint-default').style.display = 'block';
      applyOpacity();
    }
  });

  const g = svg.append('g').attr('transform', `translate(${W/2},${H/2})`);

  // Ribbons
  g.append('g').selectAll('path')
    .data(chord).join('path')
    .attr('class', 'ribbon')
    .attr('d', ribbon)
    .attr('fill',    d => getColor(d.source.index, matrix, state.colorBy))
    .attr('opacity', 0.65)
    .on('mousemove', (e, d) => {
      const si = d.source.index, ti = d.target.index;
      const fwd = matrix[si][ti], bwd = matrix[ti][si];
      showTooltip(e,
        `<div class="tt-title">${stations[si].short} ↔ ${stations[ti].short}</div>` +
        `${stations[si].short} → ${stations[ti].short}: <b>${fwd.toLocaleString()}</b> trips<br>` +
        `${stations[ti].short} → ${stations[si].short}: <b>${bwd.toLocaleString()}</b> trips<br>` +
        `Net flow: <b>${Math.abs(fwd-bwd).toLocaleString()}</b> ` +
        `${fwd > bwd ? '→ ' + stations[ti].short : '→ ' + stations[si].short}`
      );
      if (state.pinned === null) {
        d3.selectAll('.ribbon').attr('opacity', rd =>
          (rd.source.index===si||rd.target.index===si||
           rd.source.index===ti||rd.target.index===ti) ? 0.8 : 0.05);
      }
    })
    .on('mouseleave', () => { hideTooltip(); if (state.pinned===null) applyOpacity(); });

  // Arcs
  const arcG = g.append('g').selectAll('g').data(chord.groups).join('g').attr('class','arc');

  arcG.append('path')
    .attr('class', 'arc-path')
    .attr('d', arc)
    .attr('fill',         (d, i) => getColor(i, matrix, state.colorBy))
    .attr('stroke',       '#0c0e1a')
    .attr('stroke-width', 1.5)
    .attr('opacity',      0.9)
    .style('cursor', 'pointer')
    .on('mousemove', (e, d) => {
      const i = d.index, st = stations[i];
      showTooltip(e,
        `<div class="tt-title">${st.short}</div>` +
        `${st.group}<br>` +
        `Trips out (shown): <b>${d3.sum(matrix[i]).toLocaleString()}</b><br>` +
        `Trips in (shown):  <b>${d3.sum(matrix.map(r => r[i])).toLocaleString()}</b><br>` +
        `All-station total: <b>${st.total.toLocaleString()}</b>`
      );
    })
    .on('mouseleave', () => { hideTooltip(); if (state.pinned===null) applyOpacity(); })
    .on('click', (e, d) => {
      e.stopPropagation();
      const i = d.index;
      if (state.pinned === i) {
        state.pinned = null; currentHighlightGroup = null;
        document.getElementById('stat-card').classList.remove('visible');
        document.getElementById('stat-hint-default').style.display = 'block';
        applyOpacity(); return;
      }
      state.pinned = i;
      currentHighlightGroup = null;
      applyOpacity();
      showStatCard(i, matrix, stations);
    });

  // Labels
  arcG.append('text')
    .each(d => { d.angle = (d.startAngle + d.endAngle) / 2; })
    .attr('dy', '0.35em')
    .attr('transform', d =>
      `rotate(${d.angle * 180/Math.PI - 90}) translate(${R + 10}) ${d.angle > Math.PI ? 'rotate(180)' : ''}`)
    .attr('text-anchor', d => d.angle > Math.PI ? 'end' : 'start')
    .attr('font-size',   n > 15 ? 10 : 11)
    .attr('fill',        (d, i) => getColor(i, matrix, state.colorBy))
    .attr('font-weight', '600')
    .text((d, i) => stations[i].short)
    .style('pointer-events', 'none');

  // Tick marks
  arcG.selectAll('line')
    .data(d => {
      const k = (d.endAngle - d.startAngle) / d.value;
      return d3.range(0, d.value, 200).map(v => ({ angle: v*k + d.startAngle }));
    })
    .join('line')
    .attr('transform', d => `rotate(${d.angle * 180/Math.PI - 90}) translate(${R},0)`)
    .attr('x2', 5).attr('stroke', '#2a2d3e').attr('stroke-width', 1);
}

// Controls
document.getElementById('n-slider').addEventListener('input', function() {
  state.n = +this.value;
  document.getElementById('n-val').textContent = state.n;
  state.pinned = null;
  render();
});

document.getElementById('color-btns').addEventListener('click', e => {
  if (!e.target.dataset.val) return;
  document.querySelectorAll('#color-btns .btn').forEach(b => b.classList.remove('active'));
  e.target.classList.add('active');
  state.colorBy = e.target.dataset.val;
  render();
});

document.getElementById('loop-btns').addEventListener('click', e => {
  if (!e.target.dataset.val) return;
  document.querySelectorAll('#loop-btns .btn').forEach(b => b.classList.remove('active'));
  e.target.classList.add('active');
  state.showLoops = e.target.dataset.val === 'true';
  render();
});

render();
</script>
</body>
</html>"""

with open("chord.html", "w", encoding="utf-8") as f:
    f.write(html)
print("Saved → chord.html")

Saved → chord.html


In [35]:
alt.data_transformers.enable('default', max_rows=None)

DataTransformerRegistry.enable('default')

In [50]:
# scatterplot, repeating data setup from groupmates
df['starttime']    = pd.to_datetime(df['starttime'])
df['trip_minutes'] = df['tripduration'] / 60
df['User Group']       = df['usertype'].map({'Subscriber': 'Member', 'Customer': 'Non-Member'})
df['Gender']           = df['gender'].map({0: 'Unknown', 1: 'Male', 2: 'Female'})
df['birth_year_clean'] = pd.to_numeric(df['birth year'], errors='coerce')
df['age']              = 2016 - df['birth_year_clean']

def age_group(a):
    if pd.isna(a) or a < 10 or a > 100: return 'Unknown'
    if a <= 21: return 'Gen Z'
    if a <= 36: return 'Millennial'
    if a <= 52: return 'Gen X'
    return 'Boomer+'

df['Age Group'] = df['age'].apply(age_group)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

df['distance_km'] = haversine(
    df['start station latitude'], df['start station longitude'],
    df['end station latitude'],   df['end station longitude']
)
df['speed_kmh'] = df['distance_km'] / (df['trip_minutes'] / 60)

def assign_neighborhood(lat, lon):
    if lat > 42.370:                    return 'North Cambridge'
    if lat > 42.355 and lon < -71.095: return 'Harvard/Allston'
    if lat > 42.355 and lon < -71.060: return 'Cambridge/MIT'
    if lat > 42.355:                   return 'Beacon Hill/West End'
    if lon  < -71.095:                 return 'Fenway/Kenmore'
    if lat > 42.348 and lon > -71.075: return 'Downtown/Financial'
    if lat > 42.348:                   return 'Back Bay'
    return 'South End/Other'

df['Neighborhood'] = df.apply(
    lambda r: assign_neighborhood(
        r['start station latitude'], r['start station longitude']), axis=1)

df = df[
    (df['distance_km']  >  0.1) &
    (df['distance_km']  < 10.0) &
    (df['trip_minutes'] >  1.0) &
    (df['trip_minutes'] < 90.0) &
    (df['speed_kmh']    < 35.0)
]

sample_per_group = 3000
sampled = (
    df.groupby('User Group', group_keys=False)
      .apply(lambda g: g.sample(min(len(g), sample_per_group), random_state=42))
      .reset_index(drop=True)
)

CATEGORIES = {
    'User Group':  ['Member', 'Non-Member'],
    'Gender':      ['Male', 'Female', 'Unknown'],
    'Age Group':   ['Gen Z', 'Millennial', 'Gen X', 'Boomer+', 'Unknown'],
    'Neighborhood':['Cambridge/MIT', 'Harvard/Allston', 'Back Bay',
                    'Downtown/Financial', 'Beacon Hill/West End',
                    'Fenway/Kenmore', 'North Cambridge', 'South End/Other'],
}

ALL_VALUES = [v for vals in CATEGORIES.values() for v in vals]
PALETTE = [
    '#4f8ef7','#f59e0b','#10b981','#fb7185','#a78bfa',
    '#f97316','#06b6d4','#84cc16','#e11d48','#8b5cf6',
    '#0ea5e9','#d97706','#14b8a6','#f43f5e','#7c3aed',
    '#0284c7','#b45309','#0f766e',
]
COLOR_MAP = {v: PALETTE[i % len(PALETTE)] for i, v in enumerate(ALL_VALUES)}

isoline_rows = []
for v in [5, 10, 15, 20]:
    for d in np.linspace(0.1, 10, 80):
        t = (d / v) * 60
        if t <= 90:
            isoline_rows.append({'distance_km': round(d,3),
                                 'trip_minutes': round(t,3),
                                 'speed': f'{v} km/h'})
isolines_df      = pd.DataFrame(isoline_rows)
isoline_label_df = isolines_df.groupby('speed').last().reset_index()

keep_cols = ['distance_km', 'speed_kmh', 'trip_minutes',
             'User Group', 'Gender', 'Age Group', 'Neighborhood',
             'start station name', 'end station name']
records = (sampled[keep_cols]
           .round({'distance_km':3,'speed_kmh':2,'trip_minutes':2})
           .to_dict(orient='records'))

# html
html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Bluebikes Scatter</title>
<script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
<style>
* { box-sizing:border-box; margin:0; padding:0; }
body {
  font-family:'Segoe UI', system-ui, sans-serif;
  background:#0f1117; color:#e2e8f0; padding:24px 28px;
}
h2   { font-size:21px; font-weight:700; color:#f0f4ff; margin-bottom:4px; }
.sub { font-size:13px; color:#64748b; margin-bottom:20px; line-height:1.6; }

.top-bar { display:flex; gap:20px; align-items:flex-start; flex-wrap:wrap; margin-bottom:18px; }
.radio-group { display:flex; flex-direction:column; gap:6px; }
.radio-label { font-size:11px; font-weight:600; text-transform:uppercase; letter-spacing:.7px; color:#64748b; }
.radio-btns  { display:flex; gap:6px; flex-wrap:wrap; }
.rbtn {
  padding:5px 13px; border:1px solid #1e2235; border-radius:6px;
  background:#12151f; color:#64748b; font-size:13px; cursor:pointer; transition:all .15s;
}
.rbtn:hover  { border-color:#4f8ef7; color:#c7d6ff; }
.rbtn.active { background:#1a2744; border-color:#4f8ef7; color:#93c5fd; font-weight:600; }

.main { display:flex; gap:20px; align-items:flex-start; }

.check-panel {
  background:#141627; border:1px solid #1e2235; border-radius:10px;
  padding:14px 16px; min-width:175px; flex-shrink:0;
}
.check-title {
  font-size:11px; font-weight:600; text-transform:uppercase;
  letter-spacing:.7px; color:#64748b; margin-bottom:8px;
}
.all-none { display:flex; gap:6px; margin-bottom:10px; }
.mini-btn {
  font-size:11px; padding:3px 9px; border:1px solid #1e2235; border-radius:4px;
  background:#0f1117; color:#64748b; cursor:pointer; transition:all .15s;
}
.mini-btn:hover { border-color:#4f8ef7; color:#93c5fd; }

.check-item {
  display:flex; align-items:center; gap:8px;
  margin-bottom:8px; font-size:13px; cursor:pointer; user-select:none;
}
.check-box {
  width:14px; height:14px; border-radius:3px; border:2px solid #2e3349;
  flex-shrink:0; transition:all .15s; position:relative;
}
.check-box.on  { border-color:var(--col); background:var(--col); }
.check-box.on::after {
  content:''; position:absolute; left:3px; top:1px;
  width:5px; height:8px; border:2px solid #fff;
  border-left:none; border-top:none; transform:rotate(45deg);
}
.check-val { transition:color .15s; }
</style>
</head>
<body>

<h2>Riding Efficiency: Trip Duration vs Distance</h2>
<p class="sub">
  Each point = one trip &nbsp;|&nbsp; Dashed = constant speed reference (km/h)<br>
  Solid lines = regression per group
</p>

<div class="top-bar">
  <div class="radio-group">
    <span class="radio-label">Color by</span>
    <div class="radio-btns" id="cat-btns">
      <button class="rbtn active" data-cat="User Group">User Group</button>
      <button class="rbtn"        data-cat="Gender">Gender</button>
      <button class="rbtn"        data-cat="Age Group">Age Group</button>
      <button class="rbtn"        data-cat="Neighborhood">Neighborhood</button>
    </div>
  </div>
</div>

<div class="main">
  <div class="check-panel">
    <div class="check-title">Show / Hide</div>
    <div class="all-none">
      <button class="mini-btn" id="btn-all">All</button>
      <button class="mini-btn" id="btn-none">None</button>
    </div>
    <div id="checkboxes"></div>
  </div>
  <div id="chart"></div>
</div>

<script>
const DATA       = DATAPLACEHOLDER;
const ISOLINES   = ISOLINESPLACEHOLDER;
const ISO_LABELS = ISOLABELSPLACEHOLDER;
const CATEGORIES = CATEGORIESPLACEHOLDER;
const COLOR_MAP  = COLORMAPPLACEHOLDER;

let state = {
  cat:     'User Group',
  visible: new Set(CATEGORIES['User Group'])
};

// ── Checkboxes ────────────────────────────────────────────────────────────────
function buildCheckboxes() {
  const el   = document.getElementById('checkboxes');
  el.innerHTML = '';
  CATEGORIES[state.cat].forEach(val => {
    const col     = COLOR_MAP[val] || '#94a3b8';
    const isOn    = state.visible.has(val);
    const wrap    = document.createElement('div');
    wrap.className = 'check-item';

    const box = document.createElement('div');
    box.className = 'check-box' + (isOn ? ' on' : '');
    box.style.setProperty('--col', col);

    const label = document.createElement('span');
    label.className = 'check-val';
    label.textContent = val;
    label.style.color = isOn ? col : '#64748b';

    wrap.appendChild(box);
    wrap.appendChild(label);
    wrap.addEventListener('click', () => {
      if (state.visible.has(val)) {
        state.visible.delete(val);
        box.classList.remove('on');
        label.style.color = '#64748b';
      } else {
        state.visible.add(val);
        box.classList.add('on');
        label.style.color = col;
      }
      render();
    });
    el.appendChild(wrap);
  });
}

document.getElementById('btn-all').addEventListener('click', () => {
  state.visible = new Set(CATEGORIES[state.cat]);
  buildCheckboxes(); render();
});
document.getElementById('btn-none').addEventListener('click', () => {
  state.visible = new Set();
  buildCheckboxes(); render();
});

// ── Spec builder ──────────────────────────────────────────────────────────────
function buildSpec() {
  const cat     = state.cat;
  const visible = [...state.visible];
  const pts     = DATA.filter(d => visible.includes(d[cat]));

  const colorScale = {
    domain: visible,
    range:  visible.map(v => COLOR_MAP[v] || '#94a3b8')
  };

  const axisStyle = {
    gridColor:'#1e2235', domainColor:'#2a2d3e',
    labelColor:'#94a3b8', titleColor:'#94a3b8',
    labelFontSize:11, titleFontSize:12
  };

  // isolines
  const isoLayer = {
    data: { values: ISOLINES },
    mark: { type:'line', strokeDash:[5,4], strokeWidth:1.2, opacity:0.35, color:'#475569' },
    encoding: {
      x:      { field:'distance_km',  type:'quantitative', axis: axisStyle },
      y:      { field:'trip_minutes', type:'quantitative', axis: axisStyle },
      detail: { field:'speed', type:'nominal' }
    }
  };

  const isoLabelLayer = {
    data: { values: ISO_LABELS },
    mark: { type:'text', align:'left', dx:5, dy:-4, fontSize:9, fontStyle:'italic', color:'#475569' },
    encoding: {
      x:    { field:'distance_km',  type:'quantitative' },
      y:    { field:'trip_minutes', type:'quantitative' },
      text: { field:'speed',        type:'nominal' }
    }
  };

  // points
  const ptLayer = {
    data: { values: pts },
    mark: { type:'circle', opacity:0.2, size:18 },
    encoding: {
      x: { field:'distance_km',  type:'quantitative',
           title:'Straight-line distance (km)', scale:{ domain:[0,10] }, axis: axisStyle },
      y: { field:'trip_minutes', type:'quantitative',
           title:'Trip duration (min)', scale:{ domain:[0,90] }, axis: axisStyle },
      color: {
        field: cat, type:'nominal', title:'Group',
        scale: colorScale,
        legend:{ orient:'bottom', titleFontSize:12, labelFontSize:11,
                 symbolOpacity:1, labelColor:'#94a3b8', titleColor:'#94a3b8' }
      },
      tooltip:[
        { field:'start station name', type:'nominal',      title:'From' },
        { field:'end station name',   type:'nominal',      title:'To' },
        { field:'distance_km',        type:'quantitative', title:'Distance (km)',  format:'.2f' },
        { field:'trip_minutes',       type:'quantitative', title:'Duration (min)', format:'.1f' },
        { field:'speed_kmh',          type:'quantitative', title:'Speed (km/h)',   format:'.1f' },
        { field:cat,                  type:'nominal',      title:'Group' },
      ]
    }
  };

  // one regression line per visible group
  const regLayers = visible.map(val => ({
    data: { values: pts.filter(d => d[cat] === val) },
    mark: { type:'line', strokeWidth:2.5, opacity:0.9, color: COLOR_MAP[val] || '#94a3b8' },
    transform: [{ regression:'trip_minutes', on:'distance_km', method:'linear' }],
    encoding: {
      x: { field:'distance_km',  type:'quantitative' },
      y: { field:'trip_minutes', type:'quantitative' }
    }
  }));

  return {
    $schema:'https://vega.github.io/schema/vega-lite/v5.json',
    width:700, height:460,
    background:'transparent',
    layer: [isoLayer, isoLabelLayer, ptLayer, ...regLayers],
    config: { view:{ stroke:'transparent' } }
  };
}

function render() {
  vegaEmbed('#chart', buildSpec(), { actions:false, renderer:'svg' });
}

// ── Category switch ───────────────────────────────────────────────────────────
document.getElementById('cat-btns').addEventListener('click', e => {
  const cat = e.target.dataset.cat;
  if (!cat) return;
  document.querySelectorAll('.rbtn').forEach(b => b.classList.remove('active'));
  e.target.classList.add('active');
  state.cat     = cat;
  state.visible = new Set(CATEGORIES[cat]);
  buildCheckboxes();
  render();
});

buildCheckboxes();
render();
</script>
</body>
</html>"""

# Inject data
html = html.replace('DATAPLACEHOLDER',       json.dumps(records))
html = html.replace('ISOLINESPLACEHOLDER',   json.dumps(isoline_rows))
html = html.replace('ISOLABELSPLACEHOLDER',  json.dumps(isoline_label_df.to_dict(orient='records')))
html = html.replace('CATEGORIESPLACEHOLDER', json.dumps(CATEGORIES))
html = html.replace('COLORMAPPLACEHOLDER',   json.dumps(COLOR_MAP))

with open('scatter.html', 'w', encoding='utf-8') as f:
    f.write(html)
print("Saved → scatter.html")

C:\Users\brint\AppData\Local\Temp\ipykernel_5196\217834947.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), sample_per_group), random_state=42))


Saved → scatter.html
